# DWthon Adaptive Planning: Dzień 2

## Jak LLM zmienia plan po tym, co odkrywa?

Wczoraj zobaczyliśmy, dlaczego pętla jest potrzebna. Dzisiaj otwieramy środek konceptu "Mojego Robota" - przynajmniej częściowo zobaczysz, o co chodzi. Nie budujemy frameworka agentowego dla samego frameworka. Budujemy mały silnik, który po każdym wyniku potrafi odpowiedzieć na jedno pytanie: "czy mój plan nadal prowadzi do celu?".

Na koniec dnia uruchomisz robota działającego bez interfejsu, który odpowie na zadane pytanie.: "Czy warto zrobić spike AI code review?". Zacznie od prostego pytania "czy warto?", a skończy z dużo lepszą odpowiedzią: "w jakim zakresie warto to zbadać i co mierzyć?".

## Zanim zaczniemy: jedna pętla, nie siedem magicznych promptów

Wiele przykładów agentów wygląda tak: prompt, *tool call*, kolejny prompt, kolejny *tool call*. To jeszcze nie tłumaczy, kto trzyma stan, kto decyduje o planie i dlaczego model przestał robić jedną rzecz, a zaczął drugą. U nas każda tura ma ten sam kształt. Dzięki temu można ją testować, odtworzyć i pokazać na osi decyzji. To jest prostsze niż budowanie wielu specjalnych agentów i dużo bliższe temu, jak później będziesz projektować system produkcyjny.

## Wczytujemy  biblioteki

In [ ]:
from dotenv import load_dotenv
import day2_runtime

from day2_runtime import (
    AddStep, DropStep, NextStep, PlanPatch, PlanStep, SearchHn,
    allowed_actions, apply_patch, commit_turn, decision_model_for,
    initial_state, render_state_for_llm, run_turn, search_hn, track,
)

load_dotenv()

print("✅ Biblioteki gotowe. Dzisiaj model zwraca decyzję, nie prozę.")

## Dwie ścieżki, ten sam robot

Możesz dziś pracować na dwóch wysokościach. Najpierw uruchamiasz krótkie kawałki kodu i patrzysz, co chronią. Potem jedna pętla składa je w robota. Nie musisz zapamiętać wszystkich klas. Wystarczy, że przy każdej z nich umiesz powiedzieć: kto tu podejmuje decyzję, a kto sprawdza granicę?

To ten sam silnik, który jutro zobaczysz w Gradio. Dzisiaj celowo nie chowamy go za interfejsem.

## Krok 0: kontrakt, zanim powstanie plan

Użytkownik pisze: "Czy warto zrobić spike AI code review?". To za mało, aby rozsądnie badać temat, ale wystarczająco dużo, aby robot przygotował propozycję kontraktu. W kontrakcie jest decyzja, horyzont, standard dowodu, budżet i frazy wyszukiwania.

Każde pole ma pochodzenie: `user`, `inferred` albo `default`. To drobiazg, który robi ogromną różnicę w zaufaniu. Robot może sensownie założyć horyzont 90 dni. Nie może udawać, że użytkownik mu go podał. Używamy tej samej struktury, z którą pracuje później pętla.

In [ ]:
contract_state = initial_state("Czy warto zrobić spike AI code review?", horizon_days=90)
contract = contract_state.contract

for name, field in vars(contract).items():
    print(f"{name:18} = {field.value!r:75} [{field.source}]")


### Plan v1: pytania badawcze, nie lista akcji

Plan robota nie brzmi: "wyszukaj, przeczytaj, podsumuj". To byłby *pipeline* (z góry określony plan kroków, które należy wykonać.). Plan zawiera pytania: jakie świeże sygnały widzi społeczność, czy istnieje dowód wdrożenia, jakie ograniczenia się powtarzają, jaki mały eksperyment zmniejszy ryzyko. Każde pytanie ma stabilne ID. To pozwala odróżnić prawdziwą zmianę planu od sytuacji, w której model przepisuje cały tekst innymi słowami. Za chwilę zobaczysz, jak `PlanPatch` dodaje, zamyka lub porzuca konkretne pytania.

In [ ]:
plan_v1 = [
    PlanStep(id="s1", question="Jakie świeże sygnały i problemy zgłasza społeczność?"),
    PlanStep(id="s2", question="Czy poza opiniami istnieje dowód realnego zastosowania?"),
    PlanStep(id="s3", question="Jakie narzędzia są dostępne?"),
    PlanStep(id="s4", question="Jaki mały eksperyment zmniejszy ryzyko decyzji?"),
]
plan_v1

## Od wyniku narzędzia do dowodu

[Hacker News](https://news.ycombinator.com) jest naszym radarem. Mówi, co przyciąga uwagę technicznej społeczności, gdzie są dyskusje i do jakich źródeł prowadzą linki. HN nie jest jednak źródłem prawdy. Dlatego robot może przeczytać wątek, ale gdy potrzebuje dowodu, pobiera README, dokumentację albo opis wdrożenia. Model wykonuje ekstrakcję do ustrukturyzowanego `Evidence`: twierdzenie, cytat, URL i rodzaj dowodu. Kod robi coś bardziej nudnego, ale koniecznego: sprawdza, czy cytat naprawdę jest w źródle, czy data wyniku mieści się w horyzoncie i czy nie dodajemy duplikatu.

### To jest prawdziwe wywołanie publicznego API Hacker News 👇

Po uruchomieniu kod wysyła zapytanie do API, wyszukuje najnowsze wpisy pasujące do podanego hasła i zwraca je w uporządkowanej, łatwej do dalszego wykorzystania formie.

In [ ]:
results = search_hn("AI code review")
for item in results[:7]:
    print(f"{item['created_at'][:10]} | {item['points']:>4} pkt | {item['title']}")

### Mini eksperyment: cytat to nie parafraza

Za chwilę celowo podmienimy cytat na ładnie brzmiącą parafrazę. Człowiek mógłby jej uwierzyć. Model też może ją stworzyć. Kod odrzuci ją natychmiast, bo nie znajdzie dokładnego fragmentu w źródle. To jest ważna granica!  Model ma interpretować, co znaczy dowód. Nie ma sam decydować, czy dowód istnieje. 

Dzięki temu hasło "bez fejków" nie jest tylko **prośbą** w prompcie (kojarzysz ten żart - dodaj na koniec **"No mistakes!"**?), tylko właściwością systemu.

#### Ten test nie potrzebuje modelu ani sieci: sprawdza prosty, twardy fakt.

In [ ]:
source_text = "Small pull requests with clear tests are easier to review."
quote = "Clear tests make review easier."  # Parafraza: brzmi sensownie, ale nie jest cytatem.

if quote in source_text:
    print("✓ Evidence przyjęte: cytat istnieje w źródle.")
else:
    print("✗ Evidence odrzucone: cytat nie występuje dosłownie w źródle.")


## NextStep i dispatcher: gdzie mieszka kontrola

`NextStep` jest decyzją modelu. Dispatcher jest kodem, który tę decyzję przyjmuje albo odrzuca. Nie są konkurencją. Jeden bez drugiego nie tworzy systemu poziomu 5. 
Model może powiedzieć: "po tych wynikach zamykam s2, dodaję pytanie o zakres pilota i pobieram README". Dispatcher sprawdza: czy `s2` istnieje i nadal jest otwarte, czy nie ma duplikatu nowego pytania, czy wybrana akcja jest teraz dozwolona i czy budżet jeszcze istnieje. Dopiero potem wykonuje akcję.

```text
stan + ostatnia obserwacja
           |
           v
  LLM: NextStep(plan_patch, next_action)
           |
           v
  dispatcher: waliduj -> wykonaj -> zapisz obserwację
           |
           v
  nowy stan + historia Turn
```

Jeśli dispatcher odrzuci decyzję, pętla nie wybucha wyjątkiem. Odrzucenie staje się kolejną obserwacją: robot widzi powód i podejmuje następną decyzję. To właśnie odróżnia ochronę systemu od ślepego `raise Error`.

### Udajemy odpowiedź LLM tylko w tej jednej komórce, aby bez kosztu zobaczyć bramkę.

In [ ]:
guardrail_state = initial_state("Czy warto zrobić spike AI code review?", horizon_days=90)
guardrail_state.plan[1].status = "closed"
invalid_patch = PlanPatch(
    reason="Próba zmiany kroku, który jest już zamknięty",
    operations=[DropStep(op="drop", step_id="s2", reason="to tylko test bramki")],
)

demo_decision = NextStep(
    situation_analysis="Chcę zmienić zamknięty krok",
    current_priority="s2",
    plan_patch=invalid_patch,
    next_action=SearchHn(action="search_hn", query="AI code review"),
)
observation, _ = commit_turn(guardrail_state, demo_decision, view="krótki test")


print(observation["reason"])
print("To odrzucenie zapisano w historii — robot dostałby je w następnej turze.")

### Kto podejmuje jaką decyzję?

| Model decyduje | Kod gwarantuje |
| --- | --- |
| które pytanie jest teraz najważniejsze | kontrakt zmienia tylko człowiek |
| czy plan wymaga patcha | patch nie dotyka zamkniętego kroku |
| którą akcję wybrać | akcja musi być w aktualnym `Union` |
| kiedy zapytać człowieka | cytaty, daty i budżet są sprawdzone |
| jaką rekomendację uzasadnić | zbyt mocna rekomendacja nie przejdzie bramki |

Nie kontrolujemy trasy. Kontrolujemy pojazd. To zdanie ma być dla Ciebie testem każdej architektury agentowej, którą zobaczysz po tym DWthonie.

## Popraktykujmy

Nie uruchamiamy tu długiego, nieprzewidywalnego przebiegu. Jest efektowny, ale... łatwo wtedy zgubić mechanizm. Kluczowe elementy. Poniższe trzy krótkie sytuacje pokazują go czyściej: agent zmienia pytanie po danych, broni raportu przed fałszywym cytatem i nie może podjąć zbyt mocnej decyzji bez podstaw. Przynajmniej do tego dazymy.

## 🧠 Zadanie 2.0: zmień plan, nie cel

Jesteś robotem dokładnie po odkryciu ważnego faktu: zespół nie ma problemu z wyborem narzędzia, tylko z tym, że AI zasypuje review komentarzami. Za chwilę masz powiedzieć liderowi zespołu, **od czego zacząć bez przepalania miesiąca pracy**.

Lista dostępnych narzędzi brzmi ciekawie, ale nie zmniejsza tego ryzyka. Potrzebujesz pytania, które doprowadzi do małego, mierzalnego pilota. To jest moment, w którym robot przestaje zbierać ciekawostki, a zaczyna pomagać podjąć decyzję.

W planie `s3` oznacza teraz: `Jakie narzędzia są dostępne?`. To rozsądne pytanie na początku researchu, ale po odkryciu problemu z szumem już nie jest najważniejsze. Sam katalog narzędzi nie powie liderowi, czy warto uruchomić pilot.

W kolejnej komórce wpisujesz zwykłe zdania w dwa pola:

1. `drop_reason` — dlaczego pytanie o narzędzia przestało być teraz przydatne;
2. `pilot_question` — nowe pytanie, które lepiej pomoże zdecydować, jak zacząć pilot.

Nie piszesz kodu ani nowego planu. Wypełniasz tylko dwa pola po polsku. Potem uruchom komórkę. Zobaczysz konkretny diff: co wypadło z planu i jakie pytanie pojawiło się zamiast tego.

Przykładowa forma odpowiedzi: `"Katalog narzędzi nie mówi, czy pilot ograniczy szum."` oraz `"Dla jakich klas zmian AI code review zmniejsza szum zamiast go zwiększać?"`. Możesz użyć dokładnie tych zdań albo napisać własne, podobne.

In [ ]:
# Najpierw uruchom jedną komórkę z importami na górze notebooka.
# W polach poniżej wpisujesz zdania po polsku, między zwykłymi cudzysłowami.
# Możesz zacząć od przykładu z tekstu nad komórką.
drop_reason = "???"
pilot_question = "???"

if "???" in {drop_reason, pilot_question}:
    print("Uzupełnij oba pola — komórka jest bezpieczna, nic jeszcze nie zmieniliśmy.")
else:
    exercise_state = initial_state("Czy warto zrobić spike AI code review?", horizon_days=90)
    my_patch = PlanPatch(
        reason="Nowa obserwacja zmienia to, co najbardziej zmniejszy ryzyko",
        operations=[
            DropStep(op="drop", step_id="s3", reason=drop_reason),
            AddStep(op="add", question=pilot_question, reason="sygnał i szum"),
        ],
    )
    print("PLAN PRZED: katalog narzędzi → decyzja")
    print("PLAN PO:    zakres pilota i miara szumu → decyzja\n")
    for change in apply_patch(exercise_state, my_patch):
        print(change)
    print([(step.id, step.status, step.question) for step in exercise_state.plan])


<details>
<summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 18px; font-weight: 900;">👉 Kliknij tutaj, aby zobaczyć podpowiedź 👈</summary>

`s3` nie jest błędne technicznie. Po prostu przestaje być najważniejsze dla tej decyzji. Dobry powód może brzmieć: "Katalog narzędzi nie mówi, czy pilot ograniczy ryzyko i szum".

Nowe pytanie zacznij od: "Dla jakich klas zmian...". Pomyśl o małych PR-ach, testach albo zmianach rutynowych.

<details>
<summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 18px; font-weight: 900;">👉 Kliknij tutaj, aby zobaczyć rozwiązanie 👈</summary>

```python
drop_reason = "Katalog narzędzi nie odpowiada, czy pilot ograniczy szum i ryzyko."
pilot_question = "Dla jakich klas zmian AI code review zmniejsza szum zamiast go zwiększać?"
```

Po uruchomieniu zobaczysz krok `s3` ze statusem `dropped` oraz nowy krok `s4`. To nie jest nowy plan od zera: zmieniły się tylko dwa konkretne elementy.

</details>
</details>

## 🧠 Zadanie 2.1: nie pozwól modelowi stworzyć fałszywego dowodu

Robot ma za chwilę oddać raport liderowi. Model proponuje twierdzenie: `"Małe PR-y z testami są łatwiejsze do review"` i chce podeprzeć je cytatem. Problem: model potrafi napisać zdanie, które **wiernie oddaje sens** źródła, ale nigdy w nim nie padło. Gdy raport pokaże je jako cytat, człowiek nie będzie mógł sprawdzić, skąd wzięła się rekomendacja.

Przykład: źródło mówi `"Small pull requests with clear tests are easier to review."`. Zdanie `"Clear tests make review easier."` znaczy prawie to samo, ale nie jest cytatem. Cytatem może być `"Small pull requests"`, bo te słowa występują dokładnie w źródle. W komórce niżej podmień `quote` na krótki, dosłowny fragment `source_text` i uruchom test. Jeśli bramka go przyjmie, raport może pokazać twierdzenie wraz ze sprawdzalnym dowodem. Jeśli odrzuci, robot nie kończy pracy: wraca do źródła albo prosi model o inny cytat.

Ten prosty `if` jest tylko najmniejszą wersją bramki. W prawdziwym systemie chowamy go w funkcji typu `validate_evidence()`, która dodatkowo może sprawdzić URL, datę, długość cytatu i duplikaty. Nie piszemy takiego `if` ręcznie przy każdym raporcie — budujemy go raz i dispatcher wywołuje go zawsze, gdy model zgłasza dowód. Wartość jest prosta: model interpretuje materiał, ale kod nie pozwala mu udawać, że ma dowód. Dzięki temu raport daje się zakwestionować i sprawdzić, zamiast wymagać wiary w ładnie brzmiący tekst.

<details>
<summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 18px; font-weight: 900;">👉 Kliknij tutaj, aby zobaczyć podpowiedź 👈</summary>

Nie pisz własnego zdania ani lepszej parafrazy. Skopiuj kilka słów dokładnie tak, jak występują w `source_text`. Wielkość liter i kolejność słów mają znaczenie.

<details>
<summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 18px; font-weight: 900;">👉 Kliknij tutaj, aby zobaczyć rozwiązanie 👈</summary>

```python
quote = "Small pull requests"
```

Teraz `validate_evidence()` zwróci `True`: raport może pokazać cytat obok twierdzenia, a lider może go sprawdzić. Parafraza może pozostać podsumowaniem modelu, ale nie może udawać dosłownego dowodu.

</details>
</details>

In [ ]:
# To uproszczona wersja jednej bramki, którą w systemie wywołuje dispatcher.
source_text = "Small pull requests with clear tests are easier to review."
claim = "Małe PR-y z testami są łatwiejsze do review."
quote = "Clear tests make review easier."  # Podmień na dosłowny fragment.

def validate_evidence(quote, source_text):
    return quote in source_text

if validate_evidence(quote, source_text):
    print(f"✓ Dowód przyjęty. Raport może użyć twierdzenia: {claim}")
    print(f"  Sprawdzalny cytat: {quote!r}")
else:
    print("✗ Dowód odrzucony. To parafraza, nie cytat.")
    print("  Następny krok robota: znajdź dosłowny cytat albo nie używaj tego jako dowodu.")


## 🧠 Zadanie 2.2: nie pozwól robotowi podpisać decyzji za Ciebie

Robot brzmi bardzo pewnie: `"działajmy"`. Ma jednak tylko opinie z HN, bez źródła pierwotnego i bez kontrargumentu. To właśnie w takim momencie ładne demo zamienia się w ryzyko biznesowe. Nie każemy modelowi być ostrożniejszy. Kod ma postawić bramkę: mocna rekomendacja wymaga mocnych podstaw. Robot może dalej szukać dowodów albo uczciwie powiedzieć "brak podstaw", ale nie może podpisać decyzji za człowieka.

Przykład: komunikat `"mamy trzy pozytywne opinie"` brzmi obiecująco, ale nie wystarcza, by powiedzieć `"działaj"`. To jak rekomendacja zakupu auta po trzech dobrych komentarzach, bez jazdy próbnej i bez sprawdzenia awarii. W kolejnej komórce wpisz w `attempted_recommendation` rekomendację, którą dispatcher powinien zatrzymać. Uruchom ją i zobacz, co dokładnie wraca do robota jako obserwacja.

In [ ]:
# Zadanie jest samowystarczalne: nie wymaga uruchomienia wcześniejszych komórek.
attempted_recommendation = "???"  # Wpisz jedną rekomendację małymi literami.
evidence = {"primary_source": 0, "counterargument": 0, "opinions": 3}

can_act = evidence["primary_source"] >= 1 and evidence["counterargument"] >= 1

if attempted_recommendation == "działaj" and not can_act:
    print("✗ Dispatcher zatrzymał rekomendację: 'działaj'.")
    print("Powód: masz opinie, ale brakuje źródła pierwotnego i kontrargumentu.")
    print("Nowa obserwacja dla robota: zbierz dowód albo zakończ uczciwie jako 'brak podstaw'.")
elif attempted_recommendation == "???":
    print("Wpisz rekomendację, którą chcesz przetestować.")
else:
    print("Ta bramka dotyczy najmocniejszej rekomendacji. Sprawdź, co się stanie dla 'działaj'.")


<details>
<summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 18px; font-weight: 900;">👉 Kliknij tutaj, aby zobaczyć podpowiedź 👈</summary>

W kodzie `can_act` jest fałszywe: brakuje obu rodzajów dowodu wymaganych do silnej rekomendacji. Pomyśl, która odpowiedź robota miałaby największą konsekwencję, gdyby przeszła bez sprawdzenia.

<details>
<summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 18px; font-weight: 900;">👉 Kliknij tutaj, aby zobaczyć rozwiązanie 👈</summary>

```python
attempted_recommendation = "działaj"
```

Dispatcher odrzuca `"działaj"`, ale nie kończy pracy za robota i nie wymyśla za niego rekomendacji. Zwraca fakt: brakuje dowodów. W następnej turze model może wybrać dalsze badanie albo powiedzieć uczciwie `"brak podstaw"`. To jest autonomia z granicą.

</details>
</details>

## Co już umiesz rozpoznać

Po tych trzech sytuacjach znasz trzy granice, które zmieniają czat z narzędziami w robota: plan może zmienić się po danych, dowód musi dać się sprawdzić, a silna rekomendacja potrzebuje silnych podstaw.

W Agentic AI ten prosty rdzeń rozbudowujemy o pamięć, więcej źródeł i dłuższe procesy. Najpierw jednak warto umieć wskazać dokładnie, co chroni każda z tych trzech bramek.

<div class="alert alert-block alert-danger">

<h2>✅ Zaliczenie Dnia 2</h2>

<h4>Co masz teraz zrobić? (3 kroki)</h4>

<ol>
  <li>▶️ <b>Wykonaj Zadania</b> i uruchom komórkę z poprawionym planem.</li>
  <li>📸 <b>Zrób screenshot</b> wyniku: co wypadło z planu, a co robot dodał zamiast tego.</li>
  <li>💬 <b>Wrzuć go na kanał <a href="https://dataworkshopclub.slack.com/archives/C0ADAKBRX3N">#dwthon_llm_day2_done</a></b> i dopisz: "Robot zmienił plan, bo ...".</li>
</ol>

Nie opisuj całego notebooka. Jedno zdanie o konkretnej decyzji jest dużo ciekawsze. Właśnie w takich zdaniach widać, czy plan był listą kroków, czy hipotezą.

</div>

## 🎯 Jeśli chcesz: pokaż jedną decyzję, nie cały projekt

Jeśli chcesz podzielić się dniem 2 w sieci, nie pisz: "zbudowałem agenta". To już nic nie znaczy. Pokaż jedną decyzję, którą robot podjął po danych. Możesz użyć takiego zdania:

> "Mój agent zaczął od pytania, czy warto testować AI code review. Po wynikach przestał szukać katalogu narzędzi i zaczął badać, dla jakich PR-ów ogranicza się szum. To jest dla mnie różnica między workflow a Adaptive Planning."

Dodaj baner wydarzenia z materiałów organizacyjnych, oznacz @DataWorkshop i @Vladimir Alekseichenko, a potem wyślij link lub screen na `hello@dataworkshop.eu` albo na kanał wydarzenia.

## Podsumowanie Dnia 2

Dzisiaj kod był ważniejszy niż wczoraj, ale nie chodziło o liczbę klas. Chodziło o to, żebyś zobaczył, gdzie rodzi się decyzja i gdzie system mówi "stop".

1. **Kontrakt stoi, plan się zmienia.** Człowiek określa problem i granice.
2. **Plan składa się z pytań.** Akcja narzędziowa jest tylko sposobem, żeby jedno z nich zamknąć.
3. **`Evidence` nie jest luźnym tekstem.** Ma cytat, źródło, datę i rodzaj dowodu.
4. **Kod pilnuje faktów.** Cytat, data, budżet i struktura nie są kwestią opinii modelu.
5. **LLM steruje procesem.** Łączy znaczenie kilku faktów, wybiera priorytet i proponuje patch.
6. **`NextStep` i dispatcher pracują razem.** Model decyduje, kod wykonuje albo zwraca uzasadnione odrzucenie.
7. **Prawdziwy wow to zmiana pytania.** Z "czy warto?" robot dochodzi do "w jakim zakresie i z jaką miarą?".

Jutro nie będziesz już patrzeć tylko na kod. Uruchomisz gotowego robota w Gradio, zmienisz mu założenie i sprawdzisz, czy autonomia naprawdę jest pod kontrolą.